# PEARL v2: Budgeted Peer Selection for Decentralised Representation Learning

**Paper:** *Learning Whom to Learn From: Budgeted Peer Selection for Decentralised Representation Learning*

## What is new in v2

| Change | Paper reference |
|---|---|
| Anchor-validated cross-quality `anchor_quality` | §VII.H, eq. (anchor_quality) |
| Hard-class alignment `hard_class_alignment` | §VII.I, eq. (hard_class_alignment) |
| Recency-weighted exploration (window W) | §VII.J, eq. (recency_exploration) |
| Full proposed method `pearl_full` | eq. (recommended_score) |
| Negative transfer rate logged + plotted | §XII.E, eq. (auto-102) |
| Per-client entropy from recency window | §XII.E, eq. (per_client_entropy) |

| Setting | Value |
|---|---|
| Dataset | Fashion-MNIST (full 60k / 10k) |
| Clients | 50 |
| Graph | Erdős–Rényi, p=0.15 |
| Partition | Dirichlet α=0.3 |
| Rounds | 150 |
| Seeds | 1, 2, 3 |
| Exchange | Encoder + decoder, local classifier head |

Run cells sequentially. Cell 16 is the official experiment.

## Cell 1 — Imports

In [ ]:
import copy
import random
import time
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

try:
    import networkx as nx
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "networkx"])
    import networkx as nx

try:
    from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Cell 2 — Configuration

In [ ]:
CONFIG = {
    # Experiment identity
    "seed": 1,

    # Data
    "dataset":           "fashion_mnist",
    "num_clients":       50,
    "partition":         "dirichlet",
    "dirichlet_alpha":   0.3,
    "classes_per_client": 2,        # only used for "shards" partition
    "train_subset":      None,      # None = full 60k
    "test_subset":       None,      # None = full 10k

    # Graph
    "graph_type": "erdos_renyi",
    "er_prob":    0.15,

    # Training
    "rounds":       150,
    "local_epochs": 1,
    "batch_size":   64,
    "lr":           1e-3,

    # Loss weights
    "lambda_rec": 1.0,
    "lambda_cls": 1.0,

    # Peer exchange
    "mixing_alpha":  0.3,
    "exchange_mode": "encoder_decoder_local_head",
    # Options: "full_model" | "encoder_only" | "encoder_decoder_local_head"

    # Peer selection (overridden per-method in run_baseline_suite)
    "selection_method": "pearl_full",
    # Options: "local_only" | "random_peer" | "static_peer" |
    #          "prototype_only" | "quality_only" |
    #          "prototype_quality" | "prototype_quality_exploration" |
    #          "anchor_quality" | "hard_class_alignment" | "pearl_full"

    # Selection score weights
    "beta_proto":   1.0,
    "beta_quality": 1.0,
    "beta_cost":    0.1,
    "beta_explore": 0.2,

    # Anchor-validated quality
    "anchor_size": 64,              # samples from D_k sent to each neighbour

    # Hard-class alignment
    "hard_class_threshold": 0.5,    # tau_cls: per-class acc below this = hard

    # Recency-weighted exploration window
    "explore_window": 10,           # W: sliding window size in rounds

    # Evaluation
    "eval_every": 1,
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])
print("Configuration ready. Rounds:", CONFIG["rounds"])


## Cell 3 — Dataset loading

In [ ]:
def get_dataset(name="fashion_mnist", train_subset=None, test_subset=None):
    transform = transforms.Compose([transforms.ToTensor()])

    if name == "fashion_mnist":
        train_ds = torchvision.datasets.FashionMNIST(
            root="./data", train=True,  download=True, transform=transform)
        test_ds  = torchvision.datasets.FashionMNIST(
            root="./data", train=False, download=True, transform=transform)
    elif name == "mnist":
        train_ds = torchvision.datasets.MNIST(
            root="./data", train=True,  download=True, transform=transform)
        test_ds  = torchvision.datasets.MNIST(
            root="./data", train=False, download=True, transform=transform)
    else:
        raise ValueError(f"Unknown dataset: {name}")

    num_classes = 10
    in_channels = 1
    img_size    = 28

    if train_subset is not None:
        train_ds = Subset(train_ds, list(range(min(train_subset, len(train_ds)))))
    if test_subset is not None:
        test_ds  = Subset(test_ds,  list(range(min(test_subset, len(test_ds)))))

    return train_ds, test_ds, num_classes, in_channels, img_size


train_ds, test_ds, NUM_CLASSES, IN_CHANNELS, IMG_SIZE = get_dataset(
    CONFIG["dataset"], CONFIG["train_subset"], CONFIG["test_subset"]
)
print(f"Train: {len(train_ds)}  Test: {len(test_ds)}  "
      f"Classes: {NUM_CLASSES}  Channels: {IN_CHANNELS}  Size: {IMG_SIZE}")


## Cell 4 — Client partitioning

In [ ]:
def get_targets(dataset):
    """Works for both raw torchvision datasets and Subset objects."""
    if isinstance(dataset, Subset):
        return np.array(dataset.dataset.targets)[np.array(dataset.indices)]
    return np.array(dataset.targets)


def iid_partition(dataset, num_clients):
    indices = np.arange(len(dataset))
    np.random.shuffle(indices)
    return np.array_split(indices, num_clients)


def dirichlet_partition(dataset, num_clients, num_classes, alpha=0.3, min_size=10):
    targets      = get_targets(dataset)
    idx_by_class = [np.where(targets == c)[0] for c in range(num_classes)]

    while True:
        client_indices = [[] for _ in range(num_clients)]
        for c in range(num_classes):
            idx_c = idx_by_class[c].copy()
            np.random.shuffle(idx_c)
            proportions = np.random.dirichlet(alpha * np.ones(num_clients))
            cuts = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
            for k, split in enumerate(np.split(idx_c, cuts)):
                client_indices[k].extend(split.tolist())
        if min(len(x) for x in client_indices) >= min_size:
            break

    return [np.array(x) for x in client_indices]


def shards_partition(dataset, num_clients, num_classes, classes_per_client=2):
    targets       = get_targets(dataset)
    class_indices = {c: np.where(targets == c)[0].tolist() for c in range(num_classes)}
    for c in range(num_classes):
        random.shuffle(class_indices[c])

    client_indices = [[] for _ in range(num_clients)]
    for k in range(num_clients):
        chosen = np.random.choice(num_classes, classes_per_client, replace=False)
        for c in chosen:
            take = max(1, len(class_indices[c]) // (
                num_clients * classes_per_client // num_classes + 1))
            client_indices[k].extend(class_indices[c][:take])
            class_indices[c] = class_indices[c][take:]

    remaining = [idx for c in range(num_classes) for idx in class_indices[c]]
    random.shuffle(remaining)
    for i, idx in enumerate(remaining):
        client_indices[i % num_clients].append(idx)

    return [np.array(x) for x in client_indices]


def make_partitions(dataset, config, num_classes):
    p = config["partition"]
    if p == "iid":
        return iid_partition(dataset, config["num_clients"])
    if p == "dirichlet":
        return dirichlet_partition(
            dataset, config["num_clients"], num_classes, config["dirichlet_alpha"])
    if p == "shards":
        return shards_partition(
            dataset, config["num_clients"], num_classes, config["classes_per_client"])
    raise ValueError(p)


## Cell 5 — Communication graph

In [ ]:
def make_graph(num_clients, graph_type="erdos_renyi", er_prob=0.15, seed=42):
    if graph_type == "ring":
        return nx.cycle_graph(num_clients)

    if graph_type == "erdos_renyi":
        attempt = seed
        while True:
            G = nx.erdos_renyi_graph(num_clients, er_prob, seed=attempt)
            if nx.is_connected(G):
                return G
            attempt += 1

    if graph_type == "scale_free":
        m = max(1, min(3, num_clients - 1))
        return nx.barabasi_albert_graph(num_clients, m, seed=seed)

    raise ValueError(graph_type)


## Cell 6 — Model: encoder-decoder-classifier

In [ ]:
class RepAEClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=10, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder_conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, stride=2, padding=1),  # 14x14
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),           # 7x7
            nn.ReLU(),
        )
        self.encoder_fc = nn.Linear(32 * 7 * 7, latent_dim)

        self.decoder_fc = nn.Linear(latent_dim, 32 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),          # 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(16, in_channels, 4, stride=2, padding=1), # 28x28
            nn.Sigmoid(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def encode(self, x):
        h = self.encoder_conv(x)
        h = h.view(h.size(0), -1)
        return self.encoder_fc(h)

    def decode(self, z):
        h = self.decoder_fc(z)
        h = h.view(h.size(0), 32, 7, 7)
        return self.decoder_conv(h)

    def forward(self, x):
        z      = self.encode(x)
        x_hat  = self.decode(z)
        logits = self.classifier(z)
        return x_hat, logits, z


## Cell 7 — Local training and evaluation

In [ ]:
def local_train(model, loader, config):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    total_loss, n = 0.0, 0

    for _ in range(config["local_epochs"]):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            x_hat, logits, _ = model(x)
            loss = (config["lambda_rec"] * F.mse_loss(x_hat, x)
                  + config["lambda_cls"] * F.cross_entropy(logits, y))
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
            n          += x.size(0)

    return total_loss / max(1, n)


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    ys, preds  = [], []
    rec_losses = []
    cls_losses = []

    for x, y in loader:
        x, y             = x.to(device), y.to(device)
        x_hat, logits, _ = model(x)
        pred             = logits.argmax(dim=1)
        ys.extend(y.cpu().numpy().tolist())
        preds.extend(pred.cpu().numpy().tolist())
        rec_losses.append(F.mse_loss(x_hat, x, reduction="sum").item())
        cls_losses.append(F.cross_entropy(logits, y, reduction="sum").item())

    n = len(ys)
    return {
        "accuracy":          accuracy_score(ys, preds),
        "macro_f1":          f1_score(ys, preds, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(ys, preds),
        "rec_mse":           sum(rec_losses) / max(1, n),
        "cls_loss":          sum(cls_losses) / max(1, n),
    }


@torch.no_grad()
def evaluate_clients(models, client_loaders):
    rows = []
    for k, model in enumerate(models):
        m = evaluate_model(model, client_loaders[k])
        m["client"] = k
        rows.append(m)
    return pd.DataFrame(rows)


## Cell 8 — Descriptors: prototypes, per-class accuracy, anchor set

In [ ]:
@torch.no_grad()
def compute_prototypes_and_class_acc(model, loader, num_classes):
    """
    Returns:
        prototypes : dict  c -> mean latent vector (CPU tensor)
        class_acc  : dict  c -> per-class accuracy on the loader
    """
    model.eval()
    proto_sums   = {c: None for c in range(num_classes)}
    proto_counts = {c: 0    for c in range(num_classes)}
    class_correct = {c: 0   for c in range(num_classes)}
    class_total   = {c: 0   for c in range(num_classes)}

    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        z      = model.encode(x)
        logits = model.classifier(z)
        preds  = logits.argmax(dim=1)

        for c in range(num_classes):
            mask = (y == c)
            if mask.any():
                zc = z[mask].sum(dim=0).detach().cpu()
                proto_sums[c]    = zc if proto_sums[c] is None else proto_sums[c] + zc
                proto_counts[c] += int(mask.sum().item())
                class_correct[c] += int((preds[mask] == c).sum().item())
                class_total[c]   += int(mask.sum().item())

    prototypes = {c: proto_sums[c] / proto_counts[c]
                  for c in range(num_classes) if proto_counts[c] > 0}
    class_acc  = {c: class_correct[c] / class_total[c]
                  for c in range(num_classes) if class_total[c] > 0}

    return prototypes, class_acc


@torch.no_grad()
def anchor_quality(model_j, anchor_xs, anchor_ys):
    """
    Evaluate model_j on a small anchor set (xs, ys) drawn from client k.
    Returns negative mean cross-entropy loss (higher = better).
    This is Q~_{k,j}^t in the paper (eq. anchor_quality).
    """
    model_j.eval()
    xs = anchor_xs.to(device)
    ys = anchor_ys.to(device)
    _, logits, _ = model_j(xs)
    loss = F.cross_entropy(logits, ys).item()
    return -loss   # higher is better


def sample_anchor(dataset_indices, train_ds, anchor_size):
    """
    Sample anchor_size indices at random from a client's dataset indices.
    Returns (xs, ys) as CPU tensors ready to be moved to device.
    """
    n       = len(dataset_indices)
    chosen  = np.random.choice(n, size=min(anchor_size, n), replace=False)
    idx     = dataset_indices[chosen]

    xs_list, ys_list = [], []
    for i in idx:
        x, y = train_ds[int(i)]
        xs_list.append(x)
        ys_list.append(y)

    return torch.stack(xs_list), torch.tensor(ys_list, dtype=torch.long)


def build_descriptors(models, client_loaders, num_classes):
    """
    Build per-client descriptors:
        prototypes : dict c -> latent prototype (CPU tensor)
        class_acc  : dict c -> per-class training accuracy
        hard_classes: set of class indices where acc < threshold (set at call time)
        quality    : self-quality score (kept for legacy baselines)
        rec_mse    : reconstruction MSE
    """
    descriptors = {}
    for k, model in enumerate(models):
        protos, cls_acc = compute_prototypes_and_class_acc(
            model, client_loaders[k], num_classes)
        metrics = evaluate_model(model, client_loaders[k])
        descriptors[k] = {
            "prototypes":  protos,
            "class_acc":   cls_acc,
            "quality":     -metrics["cls_loss"],   # self-quality (legacy)
            "rec_mse":     metrics["rec_mse"],
        }
    return descriptors


def get_hard_classes(class_acc, threshold):
    """Return set of classes where per-class accuracy < threshold."""
    return {c for c, acc in class_acc.items() if acc < threshold}


## Cell 9 — Peer selection (updated with three new criteria)

In [ ]:
def synthetic_comm_cost(k, j):
    """Placeholder cost proportional to node-id distance."""
    return 1.0 + 0.05 * abs(k - j)


def minmax_normalize(values, eps=1e-8):
    v    = np.array(values, dtype=float)
    vmin = v.min()
    vmax = v.max()
    if abs(vmax - vmin) < eps:
        return np.zeros_like(v)
    return (v - vmin) / (vmax - vmin + eps)


def prototype_distance_hard(proto_k, proto_j, hard_classes):
    """
    Mean squared L2 distance restricted to hard_classes ∩ shared classes.
    Falls back to all shared classes if intersection is empty.
    Returns inf if no shared classes at all.
    """
    shared = set(proto_k) & set(proto_j)
    if not shared:
        return float("inf")

    focus = shared & hard_classes if hard_classes else set()
    use   = focus if focus else shared

    dist = sum(torch.norm(proto_k[c] - proto_j[c], p=2).item() ** 2 for c in use)
    return dist / len(use)


def recency_count(selection_history, k, j):
    """Number of times k selected j within the sliding window (deque)."""
    return sum(1 for jj in selection_history[(k,)] if jj == j)


def select_peer(k, neighbors, descriptors, models_snapshot,
                client_indices_ref, train_ds_ref,
                method, selection_history, config, static_peer_cache):
    """
    Select one peer for client k.

    New parameters vs v1:
        models_snapshot    : list of snapshotted models (for anchor evaluation)
        client_indices_ref : list of np arrays of training indices per client
        train_ds_ref       : the full training dataset (for anchor sampling)
        selection_history  : dict (k,) -> deque of recently selected j values
    """
    neigh = neighbors[k]
    if not neigh:
        return None

    if method == "local_only":
        return None

    if method == "random_peer":
        return random.choice(neigh)

    if method == "static_peer":
        if k not in static_peer_cache:
            static_peer_cache[k] = random.choice(neigh)
        return static_peer_cache[k]

    # ------------------------------------------------------------------ #
    # Shared pre-computation for all composite methods
    # ------------------------------------------------------------------ #
    hard_classes = get_hard_classes(
        descriptors[k]["class_acc"],
        config["hard_class_threshold"]
    )

    # Anchor set for anchor-validated quality (sampled once per round per client)
    anchor_xs, anchor_ys = sample_anchor(
        client_indices_ref[k], train_ds_ref, config["anchor_size"]
    )

    W = config["explore_window"]

    raw_D, raw_Q, raw_AQ, raw_C, raw_E = [], [], [], [], []

    for j in neigh:
        # Prototype distance (hard-class focused where applicable)
        D = prototype_distance_hard(
            descriptors[k]["prototypes"],
            descriptors[j]["prototypes"],
            hard_classes
        )
        D = D if np.isfinite(D) else 1e6

        # Self-quality (legacy, used by old methods)
        Q = descriptors[j]["quality"]

        # Anchor-validated cross-quality: how well j performs on k's data
        AQ = anchor_quality(models_snapshot[j], anchor_xs, anchor_ys)

        # Communication cost
        C = synthetic_comm_cost(k, j)

        # Recency-weighted exploration bonus
        recent_count = sum(1 for jj in selection_history[(k,)] if jj == j)
        E = 1.0 / (1.0 + recent_count)

        raw_D.append(D)
        raw_Q.append(Q)
        raw_AQ.append(AQ)
        raw_C.append(C)
        raw_E.append(E)

    D_norm  = minmax_normalize(raw_D)
    Q_norm  = minmax_normalize(raw_Q)
    AQ_norm = minmax_normalize(raw_AQ)
    C_norm  = minmax_normalize(raw_C)
    E_norm  = minmax_normalize(raw_E)

    scores = {}
    for idx, j in enumerate(neigh):

        if method == "prototype_only":
            scores[j] = -D_norm[idx]

        elif method == "quality_only":
            # Legacy: uses self-quality
            scores[j] = Q_norm[idx]

        elif method == "prototype_quality":
            # Legacy composite
            scores[j] = (- config["beta_proto"]   * D_norm[idx]
                         + config["beta_quality"]  * Q_norm[idx])

        elif method == "prototype_quality_exploration":
            # Legacy composite + cumulative exploration
            # (kept for direct comparison with v1 results)
            scores[j] = (- config["beta_proto"]   * D_norm[idx]
                         + config["beta_quality"]  * Q_norm[idx]
                         - config["beta_cost"]     * C_norm[idx]
                         + config["beta_explore"]  * E_norm[idx])

        elif method == "anchor_quality":
            # New: anchor-validated quality only
            scores[j] = AQ_norm[idx]

        elif method == "hard_class_alignment":
            # New: hard-class prototype alignment + anchor quality
            scores[j] = (- config["beta_proto"]   * D_norm[idx]
                         + config["beta_quality"]  * AQ_norm[idx])

        elif method == "pearl_full":
            # New: full recommended score from paper eq. (recommended_score)
            # hard-class prototype + anchor quality + comm cost + recency explore
            scores[j] = (- config["beta_proto"]   * D_norm[idx]
                         + config["beta_quality"]  * AQ_norm[idx]
                         - config["beta_cost"]     * C_norm[idx]
                         + config["beta_explore"]  * E_norm[idx])

        else:
            raise ValueError(f"Unknown selection method: {method}")

    return max(scores, key=scores.get)


## Cell 10 — Model exchange and mixing

In [ ]:
def mix_state_dict(sd_k, sd_j, alpha, filter_fn=None):
    out = copy.deepcopy(sd_k)
    for name in out:
        if filter_fn is None or filter_fn(name):
            out[name] = (1 - alpha) * sd_k[name].float() + alpha * sd_j[name].float()
    return out


def exchange_and_mix(model_k, model_j, config):
    alpha = config["mixing_alpha"]
    mode  = config["exchange_mode"]
    sd_k  = model_k.state_dict()
    sd_j  = model_j.state_dict()

    if mode == "full_model":
        new_sd = mix_state_dict(sd_k, sd_j, alpha)

    elif mode == "encoder_only":
        new_sd = mix_state_dict(sd_k, sd_j, alpha,
                                lambda n: n.startswith("encoder"))

    elif mode == "encoder_decoder_local_head":
        new_sd = mix_state_dict(sd_k, sd_j, alpha,
                                lambda n: n.startswith("encoder") or n.startswith("decoder"))
    else:
        raise ValueError(mode)

    model_k.load_state_dict(new_sd)
    return model_k


## Cell 11 — Communication cost helpers

In [ ]:
def parameter_bytes(model, mode):
    total = 0
    for name, p in model.named_parameters():
        include = (
            mode == "full_model"
            or (mode == "encoder_only" and name.startswith("encoder"))
            or (mode == "encoder_decoder_local_head"
                and (name.startswith("encoder") or name.startswith("decoder")))
        )
        if include:
            total += p.numel() * 4   # float32
    return total


## Cell 12 — Selection entropy (per-client, recency-window based)

In [ ]:
def compute_selection_entropy(selection_history, neighbors, window):
    """
    Per-client normalised Shannon entropy of peer selections within the
    recent window, averaged across clients. Returns a value in [0, 1].

      1.0 = perfectly uniform (random_peer)
      0.0 = always picks the same neighbour (collapse)

    Uses the recency window counts to match the exploration term
    in the composite score, directly validating Lemma 2.
    """
    entropies = []
    for k, nbrs in neighbors.items():
        if len(nbrs) <= 1:
            continue
        counts = np.array(
            [sum(1 for jj in selection_history[(k,)] if jj == j) for j in nbrs],
            dtype=float
        )
        total = counts.sum()
        if total == 0:
            continue
        probs  = counts / total
        probs  = probs[probs > 0]
        h      = -np.sum(probs * np.log(probs))
        h_norm = h / np.log(len(nbrs))
        entropies.append(h_norm)

    return float(np.mean(entropies)) if entropies else 0.0


## Cell 13 — Negative transfer rate

In [ ]:
def compute_negative_transfer_rate(prev_val_losses, curr_val_losses):
    """
    Fraction of clients for whom validation loss increased after mixing.
    prev/curr_val_losses: dicts  k -> cls_loss on client k's own data.
    Returns a float in [0, 1].
    """
    if not prev_val_losses:
        return 0.0
    events = sum(
        1 for k in curr_val_losses
        if k in prev_val_losses and curr_val_losses[k] > prev_val_losses[k]
    )
    return events / len(curr_val_losses)


## Cell 14 — Single experiment run

In [ ]:
def run_experiment(config):
    set_seed(config["seed"])

    # Data
    train_ds, test_ds, num_classes, in_channels, _ = get_dataset(
        config["dataset"], config["train_subset"], config["test_subset"])
    client_indices = make_partitions(train_ds, config, num_classes)
    client_loaders = [
        DataLoader(Subset(train_ds, idx.tolist()),
                   batch_size=config["batch_size"], shuffle=True,
                   num_workers=2, pin_memory=True)
        for idx in client_indices
    ]
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                             num_workers=2, pin_memory=True)

    # Graph
    G         = make_graph(config["num_clients"], config["graph_type"],
                           config["er_prob"], config["seed"])
    neighbors = {k: list(G.neighbors(k)) for k in range(config["num_clients"])}

    # Models — all initialised from the same random state
    init_model = RepAEClassifier(in_channels, num_classes).to(device)
    init_sd    = init_model.state_dict()
    models     = [RepAEClassifier(in_channels, num_classes).to(device)
                  for _ in range(config["num_clients"])]
    for m in models:
        m.load_state_dict(copy.deepcopy(init_sd))

    comm_bytes_per_exchange = parameter_bytes(models[0], config["exchange_mode"])

    # Selection history: (k,) -> deque of recently selected j values
    # Using a flat deque per client; we check j membership inside select_peer
    W                  = config["explore_window"]
    selection_history  = defaultdict(lambda: deque(maxlen=W))
    static_peer_cache  = {}
    history            = []
    cumulative_bytes   = 0
    prev_val_losses    = {}   # for negative transfer tracking

    for t in range(config["rounds"]):
        t_start = time.time()

        # --- Local training ---
        local_losses = [local_train(models[k], client_loaders[k], config)
                        for k in range(config["num_clients"])]

        # --- Descriptor construction ---
        descriptors = build_descriptors(models, client_loaders, num_classes)

        # --- Per-client validation loss before exchange (for negative transfer) ---
        pre_val_losses = {k: descriptors[k]["rec_mse"] + abs(descriptors[k]["quality"])
                          for k in range(config["num_clients"])}

        # --- Peer selection and exchange ---
        selected_pairs = []
        if config["selection_method"] != "local_only":
            # Snapshot models so intra-round updates do not cascade
            snapshots = [copy.deepcopy(m) for m in models]

            for k in range(config["num_clients"]):
                j = select_peer(
                    k, neighbors, descriptors, snapshots,
                    client_indices, train_ds,
                    config["selection_method"],
                    selection_history, config, static_peer_cache
                )
                if j is not None:
                    selected_pairs.append((k, j))
                    selection_history[(k,)].append(j)   # update recency window

            for k, j in selected_pairs:
                exchange_and_mix(models[k], snapshots[j], config)

        # --- Post-exchange validation loss (for negative transfer) ---
        post_descriptors = build_descriptors(models, client_loaders, num_classes)
        post_val_losses  = {k: post_descriptors[k]["rec_mse"] + abs(post_descriptors[k]["quality"])
                            for k in range(config["num_clients"])}
        neg_transfer_rate = compute_negative_transfer_rate(pre_val_losses, post_val_losses)

        # --- Evaluation ---
        if t % config["eval_every"] == 0 or t == config["rounds"] - 1:
            client_df      = evaluate_clients(models, client_loaders)
            global_metrics = [evaluate_model(models[k], test_loader)
                              for k in range(config["num_clients"])]

            mean_global_acc  = float(np.mean([m["accuracy"]  for m in global_metrics]))
            mean_global_f1   = float(np.mean([m["macro_f1"]  for m in global_metrics]))
            mean_client_acc  = float(client_df["accuracy"].mean())
            worst_client_acc = float(client_df["accuracy"].min())
            std_client_acc   = float(client_df["accuracy"].std())

            round_bytes      = len(selected_pairs) * comm_bytes_per_exchange
            cumulative_bytes += round_bytes

            sel_entropy = (
                compute_selection_entropy(selection_history, neighbors, W)
                if config["selection_method"] != "local_only" else 0.0
            )

            row = {
                "round":                  t,
                "method":                 config["selection_method"],
                "graph":                  config["graph_type"],
                "partition":              config["partition"],
                "dirichlet_alpha":        config.get("dirichlet_alpha"),
                "exchange_mode":          config["exchange_mode"],
                "mean_local_loss":        float(np.mean(local_losses)),
                "mean_global_accuracy":   mean_global_acc,
                "mean_global_macro_f1":   mean_global_f1,
                "mean_client_accuracy":   mean_client_acc,
                "worst_client_accuracy":  worst_client_acc,
                "std_client_accuracy":    std_client_acc,
                "neg_transfer_rate":      neg_transfer_rate,
                "round_exchanges":        len(selected_pairs),
                "round_bytes":            round_bytes,
                "cum_bytes":              cumulative_bytes,
                "selection_entropy":      sel_entropy,
                "runtime_sec":            time.time() - t_start,
            }
            history.append(row)

            print(
                f"Round {t:03d} | {config['selection_method']:<30} | "
                f"acc={mean_global_acc:.3f} | f1={mean_global_f1:.3f} | "
                f"worst={worst_client_acc:.3f} | "
                f"neg_xfer={neg_transfer_rate:.3f} | "
                f"entropy={sel_entropy:.3f} | "
                f"MB={cumulative_bytes/1e6:.2f}"
            )

        prev_val_losses = post_val_losses

    return pd.DataFrame(history)


## Cell 15 — Baseline suite runner

In [ ]:
METHODS = [
    "local_only",
    "random_peer",
    "static_peer",
    "prototype_only",
    "quality_only",
    "prototype_quality",
    "prototype_quality_exploration",   # v1 composite (kept for comparison)
    "anchor_quality",                  # NEW: anchor-validated quality only
    "hard_class_alignment",            # NEW: hard-class proto + anchor quality
    "pearl_full",                      # NEW: full recommended score
]


def run_baseline_suite(base_config, methods=METHODS):
    all_results = []
    for method in methods:
        cfg = copy.deepcopy(base_config)
        cfg["selection_method"] = method
        print(f"\n{'='*60}")
        print(f"Running: {method} | graph: {cfg['graph_type']}")
        print(f"{'='*60}")
        df = run_experiment(cfg)
        all_results.append(df)
    return pd.concat(all_results, ignore_index=True)


## Cell 16 — Experiment 1: ER-150, 50 clients, 3 seeds  (THE OFFICIAL RUN)

In [ ]:
all_seed_results = []

for seed in [1, 2, 3]:
    CONFIG["seed"]            = seed
    CONFIG["num_clients"]     = 50
    CONFIG["train_subset"]    = None      # full 60k
    CONFIG["test_subset"]     = None      # full 10k
    CONFIG["rounds"]          = 150
    CONFIG["partition"]       = "dirichlet"
    CONFIG["dirichlet_alpha"] = 0.3
    CONFIG["graph_type"]      = "erdos_renyi"
    CONFIG["er_prob"]         = 0.15
    CONFIG["exchange_mode"]   = "encoder_decoder_local_head"

    print(f"\n{'#'*60}")
    print(f"# Seed {seed}")
    print(f"{'#'*60}")

    res         = run_baseline_suite(CONFIG, METHODS)
    res["seed"] = seed
    all_seed_results.append(res)

results_all = pd.concat(all_seed_results, ignore_index=True)
results_all.to_csv("pearl_er150_50clients_allseeds_v2.csv", index=False)
print("\nSaved: pearl_er150_50clients_allseeds_v2.csv")


## Cell 17 — Summary table (mean ± std across seeds, final round)

In [ ]:
final_rows = (
    results_all
    .sort_values("round")
    .groupby(["seed", "method"])
    .tail(1)
)

summary = (
    final_rows
    .groupby("method")
    .agg(
        mean_acc          =("mean_global_accuracy",  "mean"),
        std_acc           =("mean_global_accuracy",  "std"),
        mean_f1           =("mean_global_macro_f1",  "mean"),
        std_f1            =("mean_global_macro_f1",  "std"),
        mean_worst        =("worst_client_accuracy", "mean"),
        std_worst         =("worst_client_accuracy", "std"),
        mean_neg_xfer     =("neg_transfer_rate",      "mean"),
        std_neg_xfer      =("neg_transfer_rate",      "std"),
        mean_entropy      =("selection_entropy",      "mean"),
        std_entropy       =("selection_entropy",      "std"),
    )
    .reset_index()
    .sort_values("mean_f1", ascending=False)
)

print("\n=== Final-round summary (mean ± std across 3 seeds) ===")
print(summary.to_string(index=False))
summary.to_csv("pearl_er150_summary_v2.csv", index=False)


## Cell 18 — Plots

In [ ]:
METHOD_COLORS = {
    "local_only":                      "#888780",
    "random_peer":                     "#378ADD",
    "static_peer":                     "#BA7517",
    "prototype_only":                  "#1D9E75",
    "quality_only":                    "#D4537E",
    "prototype_quality":               "#7F77DD",
    "prototype_quality_exploration":   "#D85A30",
    "anchor_quality":                  "#E24B4A",   # red
    "hard_class_alignment":            "#0F6E56",   # dark teal
    "pearl_full":                      "#3C3489",   # deep purple — the proposed method
}

METHOD_WIDTHS = {m: 2.5 if m == "pearl_full" else 1.5 for m in METHOD_COLORS}


def plot_metric_vs_rounds(results, metric, ylabel, title):
    """Mean ± seed-range of a metric over rounds, all methods."""
    fig, ax = plt.subplots(figsize=(10, 5))

    for method, grp in results.groupby("method"):
        if metric not in grp.columns:
            continue
        pivot = grp.pivot_table(index="round", columns="seed", values=metric)
        mean  = pivot.mean(axis=1)
        lo    = pivot.min(axis=1)
        hi    = pivot.max(axis=1)
        color = METHOD_COLORS.get(method, "#333")
        lw    = METHOD_WIDTHS.get(method, 1.5)
        ax.plot(mean.index, mean.values, label=method, color=color, linewidth=lw)
        ax.fill_between(mean.index, lo.values, hi.values, alpha=0.10, color=color)

    ax.set_xlabel("Round")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=7, loc="lower right", ncol=2)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    fname = f"pearl_{metric}_vs_rounds.pdf"
    plt.savefig(fname, bbox_inches="tight")
    plt.show()
    print(f"Saved {fname}")


def plot_metric_vs_comm(results, metric, ylabel, title):
    """Metric vs cumulative MB, mean across seeds."""
    fig, ax = plt.subplots(figsize=(10, 5))

    for method, grp in results.groupby("method"):
        if metric not in grp.columns:
            continue
        mean_grp = grp.groupby("round")[["cum_bytes", metric]].mean().reset_index()
        color    = METHOD_COLORS.get(method, "#333")
        lw       = METHOD_WIDTHS.get(method, 1.5)
        ax.plot(mean_grp["cum_bytes"] / 1e6, mean_grp[metric],
                label=method, color=color, linewidth=lw)

    ax.set_xlabel("Cumulative communication (MB)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=7, loc="lower right", ncol=2)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    fname = f"pearl_{metric}_vs_comm.pdf"
    plt.savefig(fname, bbox_inches="tight")
    plt.show()
    print(f"Saved {fname}")


def plot_selection_entropy(results):
    """Normalised per-client selection entropy — validates Lemma 2."""
    fig, ax = plt.subplots(figsize=(10, 4))
    df = results[results["method"] != "local_only"]

    for method, grp in df.groupby("method"):
        pivot = grp.pivot_table(index="round", columns="seed",
                                values="selection_entropy")
        mean  = pivot.mean(axis=1)
        color = METHOD_COLORS.get(method, "#333")
        lw    = METHOD_WIDTHS.get(method, 1.5)
        ax.plot(mean.index, mean.values, label=method, color=color, linewidth=lw)

    ax.axhline(1.0, color="#378ADD", linestyle="--", linewidth=0.8, alpha=0.5,
               label="random_peer ceiling")
    ax.set_xlabel("Round")
    ax.set_ylabel("Normalised selection entropy")
    ax.set_ylim(0, 1.1)
    ax.set_title("Peer-selection entropy (1.0 = uniform; validates Lemma 2)")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig("pearl_selection_entropy.pdf", bbox_inches="tight")
    plt.show()
    print("Saved pearl_selection_entropy.pdf")


def plot_negative_transfer(results):
    """
    Negative transfer rate over rounds — primary empirical claim.
    Lower is better. pearl_full should lie below random_peer.
    """
    fig, ax = plt.subplots(figsize=(10, 4))
    df = results[results["method"] != "local_only"]

    for method, grp in df.groupby("method"):
        pivot = grp.pivot_table(index="round", columns="seed",
                                values="neg_transfer_rate")
        mean  = pivot.mean(axis=1)
        color = METHOD_COLORS.get(method, "#333")
        lw    = METHOD_WIDTHS.get(method, 1.5)
        ax.plot(mean.index, mean.values, label=method, color=color, linewidth=lw)

    ax.set_xlabel("Round")
    ax.set_ylabel("Negative transfer rate")
    ax.set_ylim(0, 1.05)
    ax.set_title("Negative transfer rate over rounds (lower = better; primary claim)")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig("pearl_neg_transfer.pdf", bbox_inches="tight")
    plt.show()
    print("Saved pearl_neg_transfer.pdf")


# Generate all plots
plot_metric_vs_rounds(results_all, "mean_global_macro_f1",
                      "Mean macro-F1", "Macro-F1 vs rounds (ER, 50 clients, α=0.3)")
plot_metric_vs_rounds(results_all, "mean_global_accuracy",
                      "Mean global accuracy", "Accuracy vs rounds")
plot_metric_vs_rounds(results_all, "worst_client_accuracy",
                      "Worst-client accuracy", "Worst-client accuracy vs rounds")
plot_metric_vs_comm(results_all, "mean_global_macro_f1",
                    "Mean macro-F1", "Macro-F1 vs communication (MB)")
plot_selection_entropy(results_all)
plot_negative_transfer(results_all)

print("\nAll plots saved as PDF.")
